# SQL Intermediate 2 Practice Exercises

**Estimated time:** ~5 hours (part of the ~9 hour sql-intermediate-2 level, alongside
`sql-intermediate-2-guide.ipynb`).

These follow `sql-intermediate-2-guide.ipynb` in order, and every heading names the **guide section** it
practises. Numbering starts at 2 because guide section 1 is the setup, which the cell below covers.

## How to Use This Notebook

- **Run the setup cell first.** It builds the same shop database from `../assets/sql/` and defines `q()` and
  `run()` — identical to `sql-intermediate`.
- Write your SQL between the triple quotes, then run the cell with **Shift + Enter**.
- Every cell ends with `assert` checks. You are right when it prints `OK` with no `AssertionError`.
- **Use the exact column names and aliases the instructions ask for**, and the exact `ORDER BY` — the checks
  compare positions, and without a deterministic sort there is no correct answer to compare against.
- Several exercises here ask a question the guide already answered on a *different* slice of the data.
  Re-deriving the same query on new inputs, without the guide open, is the actual practice.
- Revenue means `quantity * unit_price * (1 - discount)` throughout, and unless an exercise says otherwise,
  cancelled orders are excluded.


## Setup — Run This First

Builds the database in memory and defines `q(sql)` and `run(sql)`. Run it once; run it again after any kernel
restart.


In [1]:
import sqlite3
import pandas as pd

SQL = "assets/sql"          # the bundled schema and CSV files
TABLES = ["categories", "customers", "employees", "products",
          "orders", "order_items", "payments"]

con = sqlite3.connect(":memory:")         # the database lives in RAM -- nothing to clean up

with open(f"{SQL}/schema.sql") as f:
    con.executescript(f.read())           # creates the seven empty tables

con.execute("PRAGMA foreign_keys = ON")   # from here on, SQLite enforces the foreign keys

for table in TABLES:
    pd.read_csv(f"{SQL}/{table}.csv").to_sql(table, con, if_exists="append", index=False)
con.commit()


def q(sql):
    """Run a SELECT and hand the result back as a pandas DataFrame."""
    return pd.read_sql_query(sql, con)


def run(sql):
    """Run statements that change data or structure: CREATE, INSERT, UPDATE, DELETE."""
    con.executescript(sql)
    con.commit()


pd.set_option("display.width", 110)
pd.set_option("display.max_rows", 25)

for table in TABLES:
    print(f"{table:12s} {q(f'SELECT COUNT(*) AS n FROM {table}')['n'][0]:>4} rows")


categories      8 rows
customers      60 rows
employees      15 rows
products       40 rows
orders        300 rows
order_items   673 rows
payments      248 rows


## Exercise 2: Deduplication  *(guide section 2)*

`customers` has two duplicate emails. For each one, find which `customer_id` should survive (the **earliest**
`signup_date`, with `customer_id` as a tiebreaker) and which should be removed.

1. `pairs` — one row per duplicate email: `email`, `kept_id`, `removed_id`. Order by `email`.
2. `total_after_dedup` — a single row, single column `n`: how many customers remain once the two removed rows
   are dropped from the full 60.

Use `ROW_NUMBER() OVER (PARTITION BY email ORDER BY signup_date, customer_id)` and read off `rn = 1` and
`rn = 2`.


In [6]:
pairs_sql = """
WITH ranked AS (
    SELECT
        customer_id,
        email,
        signup_date,
        ROW_NUMBER() OVER (
            PARTITION BY email
            ORDER BY signup_date, customer_id
        ) AS rn
    FROM customers
),
pairs AS (
    SELECT
        email,
        MAX(CASE WHEN rn = 1 THEN customer_id END) AS kept_id,
        MAX(CASE WHEN rn = 2 THEN customer_id END) AS removed_id
    FROM ranked
    GROUP BY email
    HAVING COUNT(*) > 1
)
SELECT *
FROM pairs
ORDER BY email;
"""
pairs = q(pairs_sql)

total_sql = """
WITH ranked AS (
    SELECT
        customer_id,
        email,
        signup_date,
        ROW_NUMBER() OVER (
            PARTITION BY email
            ORDER BY signup_date, customer_id
        ) AS rn
    FROM customers
),
pairs AS (
    SELECT
        email,
        MAX(CASE WHEN rn = 1 THEN customer_id END) AS kept_id,
        MAX(CASE WHEN rn = 2 THEN customer_id END) AS removed_id
    FROM ranked
    GROUP BY email
    HAVING COUNT(*) > 1
)
SELECT 60 - COUNT(removed_id) AS n
FROM pairs;
"""
total_after_dedup = q(total_sql)

assert list(pairs.columns) == ["email", "kept_id", "removed_id"]
assert len(pairs) == 2
assert pairs.values.tolist() == [
    ["neha.reddy@example.com", 59, 12],
    ["tarun.khan@example.com", 28, 60],
]
assert list(total_after_dedup.columns) == ["n"]
assert total_after_dedup["n"][0] == 58
print("OK")


OK


## Exercise 3: Self-Joins for Association  *(guide section 3)*

1. `pairs` — the **top 5** product pairs by how often they appear on the same order. Columns `product_a`,
   `product_b`, `times_together`. Each unordered pair counted once (`p2.product_id > p1.product_id` in the
   `ON`). Order by `times_together` descending, then `product_a`, then `product_b`.
2. `with_partner` — a single row, single column `n`: how many distinct products have **ever** shared an order
   with at least one other product.

For part 2, count distinct products in both directions (`<>`, not `>`) — a product that only ever appears as
the *smaller* id in its pairs would otherwise be missed.


In [11]:
pairs_sql = """
SELECT
    p2.name AS product_a,
    p1.name AS product_b,
    COUNT(*) AS times_together
FROM order_items o
JOIN order_items oi
    ON o.order_id = oi.order_id
   AND o.product_id > oi.product_id
JOIN products p1
    ON o.product_id = p1.product_id
JOIN products p2
    ON oi.product_id = p2.product_id
GROUP BY
    p1.name,
    p2.name
ORDER BY
    times_together DESC,
    product_a,
    product_b
LIMIT 5;

"""
pairs = q(pairs_sql)

with_partner_sql = """
SELECT COUNT(DISTINCT product_id) AS n
FROM (
    SELECT o.product_id
    FROM order_items o
    JOIN order_items oi
        ON o.order_id = oi.order_id
       AND o.product_id > oi.product_id

    UNION

    SELECT oi.product_id
    FROM order_items o
    JOIN order_items oi
        ON o.order_id = oi.order_id
       AND o.product_id > oi.product_id
);
"""
print(pairs)
print(pairs.columns)
with_partner = q(with_partner_sql)

assert list(pairs.columns) == ["product_a", "product_b", "times_together"]
assert len(pairs) == 5
assert pairs.values.tolist() == [
    ["Braid USB-C Cable 1m", "Anchor 100W Charger", 4],
    ["Braid USB-C Cable 1m", "Clarity 24 Monitor", 4],
    ["Braid USB-C Cable 1m", "Vault 4TB HDD", 4],
    ["Clarity 32 4K Monitor", "Vault 2TB SSD", 4],
    ["Echo Buds Pro", "Rumble Bluetooth Speaker", 4],
]
assert list(with_partner.columns) == ["n"]
assert with_partner["n"][0] == 38
print("OK")


               product_a                 product_b  times_together
0   Braid USB-C Cable 1m       Anchor 100W Charger               4
1   Braid USB-C Cable 1m        Clarity 24 Monitor               4
2   Braid USB-C Cable 1m             Vault 4TB HDD               4
3  Clarity 32 4K Monitor             Vault 2TB SSD               4
4          Echo Buds Pro  Rumble Bluetooth Speaker               4
Index(['product_a', 'product_b', 'times_together'], dtype='str')
OK


## Exercise 4: Flattening a Hierarchy Without Recursion  *(guide section 4)*

Every employee, alongside their manager and their manager's manager.

Columns: `employee_id`, `employee` (the `name`), `role`, `manager`, `manager_role`, `director`,
`director_role`. Use `'none'` — not `NULL` — wherever a level does not exist (the founder has neither a
manager nor a director; the three managers have no director).

Order by `employee_id`.

Chain two `LEFT JOIN`s of `employees` to itself, then `COALESCE` every one of the four upper-level columns to
`'none'`.


In [17]:
sql = """
select e.employee_id as employee_id , e.name as employee , e.role as role ,   COALESCE(m.name, 'none') AS manager,
    COALESCE(m.role, 'none') AS manager_role,
    COALESCE(mm.name, 'none') AS director,
    COALESCE(mm.role, 'none') AS director_role
from employees e left join employees m on e.manager_id=m.employee_id
left join employees mm on m.manager_id = mm.employee_id
order by e.employee_id
"""

out = q(sql)
print(out)

assert list(out.columns) == ["employee_id", "employee", "role", "manager", "manager_role", "director", "director_role"]
assert len(out) == 15
assert out.values.tolist() == [
    [1, "Radhika Menon", "Founder", "none", "none", "none", "none"],
    [2, "Vikram Nair", "Sales Manager", "Radhika Menon", "Founder", "none", "none"],
    [3, "Sunita Rao", "Sales Manager", "Radhika Menon", "Founder", "none", "none"],
    [4, "Imran Sheikh", "Support Manager", "Radhika Menon", "Founder", "none", "none"],
    [5, "Arjun Pillai", "Sales Rep", "Vikram Nair", "Sales Manager", "Radhika Menon", "Founder"],
    [6, "Kavya Krishnan", "Sales Rep", "Vikram Nair", "Sales Manager", "Radhika Menon", "Founder"],
    [7, "Devendra Joshi", "Sales Rep", "Vikram Nair", "Sales Manager", "Radhika Menon", "Founder"],
    [8, "Priya Balan", "Sales Rep", "Sunita Rao", "Sales Manager", "Radhika Menon", "Founder"],
    [9, "Nikhil Verma", "Sales Rep", "Sunita Rao", "Sales Manager", "Radhika Menon", "Founder"],
    [10, "Farah Qureshi", "Sales Rep", "Sunita Rao", "Sales Manager", "Radhika Menon", "Founder"],
    [11, "Sanjay Gupta", "Support Agent", "Imran Sheikh", "Support Manager", "Radhika Menon", "Founder"],
    [12, "Meera Iyer", "Support Agent", "Imran Sheikh", "Support Manager", "Radhika Menon", "Founder"],
    [13, "Tarun Das", "Support Agent", "Imran Sheikh", "Support Manager", "Radhika Menon", "Founder"],
    [14, "Lakshmi Suresh", "Sales Rep", "Vikram Nair", "Sales Manager", "Radhika Menon", "Founder"],
    [15, "Omar Farooq", "Sales Rep", "Sunita Rao", "Sales Manager", "Radhika Menon", "Founder"],
]
print("OK")


    employee_id        employee             role        manager     manager_role       director  \
0             1   Radhika Menon          Founder           none             none           none   
1             2     Vikram Nair    Sales Manager  Radhika Menon          Founder           none   
2             3      Sunita Rao    Sales Manager  Radhika Menon          Founder           none   
3             4    Imran Sheikh  Support Manager  Radhika Menon          Founder           none   
4             5    Arjun Pillai        Sales Rep    Vikram Nair    Sales Manager  Radhika Menon   
5             6  Kavya Krishnan        Sales Rep    Vikram Nair    Sales Manager  Radhika Menon   
6             7  Devendra Joshi        Sales Rep    Vikram Nair    Sales Manager  Radhika Menon   
7             8     Priya Balan        Sales Rep     Sunita Rao    Sales Manager  Radhika Menon   
8             9    Nikhil Verma        Sales Rep     Sunita Rao    Sales Manager  Radhika Menon   
9         

## Exercise 5: Unpivoting  *(guide section 5)*

Take the **5 most expensive products** (by `price`) and unpivot their `price`, `cost` and `stock` into rows.

Columns `product_id`, `name`, `metric` (`'price'`, `'cost'` or `'stock'`), `value`. Order by `product_id`, then
`metric`.

Build the top-5 list once, in a CTE, and reuse it in all three `UNION ALL` branches — recomputing "the 5 most
expensive products" three times risks the three branches disagreeing if the data ever changes.


In [24]:
sql = """
with cte as (select product_id , name , price , cost ,stock from products order by price desc limit 5)
 select product_id , name , 'price' as metric , price as value from cte
union all
 select product_id , name , 'cost' as metric , cost as value from cte
union all 
select product_id , name , 'stock' as metric , stock as value from cte
order by product_id , metric
"""

out = q(sql)
print(out)

assert list(out.columns) == ["product_id", "name", "metric", "value"]
assert len(out) == 15
assert out.values.tolist() == [
    [2, "Aster 15 Pro Laptop", "cost", 78000.0],
    [2, "Aster 15 Pro Laptop", "price", 94000.0],
    [2, "Aster 15 Pro Laptop", "stock", 11.0],
    [3, "Nimbus Air Laptop", "cost", 99000.0],
    [3, "Nimbus Air Laptop", "price", 118000.0],
    [3, "Nimbus Air Laptop", "stock", 6.0],
    [5, "Vega Book 16 Studio", "cost", 121000.0],
    [5, "Vega Book 16 Studio", "price", 142000.0],
    [5, "Vega Book 16 Studio", "stock", 4.0],
    [8, "Orbit Ultra Phone", "cost", 65000.0],
    [8, "Orbit Ultra Phone", "price", 79000.0],
    [8, "Orbit Ultra Phone", "stock", 17.0],
    [36, "Frame Mirrorless Body", "cost", 71000.0],
    [36, "Frame Mirrorless Body", "price", 86000.0],
    [36, "Frame Mirrorless Body", "stock", 7.0],
]
print("OK")


    product_id                   name metric     value
0            2    Aster 15 Pro Laptop   cost   78000.0
1            2    Aster 15 Pro Laptop  price   94000.0
2            2    Aster 15 Pro Laptop  stock      11.0
3            3      Nimbus Air Laptop   cost   99000.0
4            3      Nimbus Air Laptop  price  118000.0
5            3      Nimbus Air Laptop  stock       6.0
6            5    Vega Book 16 Studio   cost  121000.0
7            5    Vega Book 16 Studio  price  142000.0
8            5    Vega Book 16 Studio  stock       4.0
9            8      Orbit Ultra Phone   cost   65000.0
10           8      Orbit Ultra Phone  price   79000.0
11           8      Orbit Ultra Phone  stock      17.0
12          36  Frame Mirrorless Body   cost   71000.0
13          36  Frame Mirrorless Body  price   86000.0
14          36  Frame Mirrorless Body  stock       7.0
OK


## Exercise 6: FIRST_VALUE and LAST_VALUE  *(guide section 6)*

One row per **category**: its cheapest and priciest product.

Columns `category` (the category's `name`), `cheapest_product`, `cheapest_price`, `priciest_product`,
`priciest_price`. Order by `category`.

Two `FIRST_VALUE`s per row (one `ORDER BY price ASC`, one `DESC`), all `PARTITION BY category_id`, then a
`ROW_NUMBER` in the same CTE to collapse each category down to a single output row.


In [37]:
sql = """
WITH cte AS (
    SELECT
        p.name AS product,
        c.category_id,
        c.name AS category,
        p.price
    FROM products p
    LEFT JOIN categories c
        ON p.category_id = c.category_id
),
ranked AS (
    SELECT
        category,

        FIRST_VALUE(product) OVER (
            PARTITION BY category_id
            ORDER BY price ASC
        ) AS cheapest_product,

        FIRST_VALUE(price) OVER (
            PARTITION BY category_id
            ORDER BY price ASC
        ) AS cheapest_price,

        FIRST_VALUE(product) OVER (
            PARTITION BY category_id
            ORDER BY price DESC
        ) AS priciest_product,

        FIRST_VALUE(price) OVER (
            PARTITION BY category_id
            ORDER BY price DESC
        ) AS priciest_price,

        ROW_NUMBER() OVER (
            PARTITION BY category_id
            ORDER BY price
        ) AS rn

    FROM cte
)
SELECT
    category,
    cheapest_product,
    cheapest_price,
    priciest_product,
    priciest_price
FROM ranked
WHERE rn = 3
ORDER BY category;
"""

out = q(sql)
print(out)

assert list(out.columns) == ["category", "cheapest_product", "cheapest_price", "priciest_product", "priciest_price"]
assert len(out) == 8
assert out.values.tolist() == [
    ["Accessories", "Braid USB-C Cable 1m", 450.0, "Clack Mechanical Keyboard", 4900.0],
    ["Audio", "Echo Buds", 3200.0, "Halo Studio Headset", 18900.0],
    ["Cameras", "Frame 50mm Lens", 15400.0, "Frame Mirrorless Body", 86000.0],
    ["Laptops", "Vega Book 13", 54000.0, "Vega Book 16 Studio", 142000.0],
    ["Monitors", "Clarity 24 Monitor", 11800.0, "Wide 34 Curved Monitor", 56000.0],
    ["Phones", "Pixi Lite Phone", 13500.0, "Orbit Ultra Phone", 79000.0],
    ["Storage", "Carry 128GB Flash Drive", 1150.0, "Vault 2TB SSD", 11200.0],
    ["Wearables", "Tick Fitness Band", 2800.0, "Tick Watch Ultra", 26500.0],
]
print("OK")


      category         cheapest_product  cheapest_price           priciest_product  priciest_price
0  Accessories     Braid USB-C Cable 1m           450.0  Clack Mechanical Keyboard          4900.0
1        Audio                Echo Buds          3200.0        Halo Studio Headset         18900.0
2      Cameras          Frame 50mm Lens         15400.0      Frame Mirrorless Body         86000.0
3      Laptops             Vega Book 13         54000.0        Vega Book 16 Studio        142000.0
4     Monitors       Clarity 24 Monitor         11800.0     Wide 34 Curved Monitor         56000.0
5       Phones          Pixi Lite Phone         13500.0          Orbit Ultra Phone         79000.0
6      Storage  Carry 128GB Flash Drive          1150.0              Vault 2TB SSD         11200.0
7    Wearables        Tick Fitness Band          2800.0           Tick Watch Ultra         26500.0
OK


## Exercise 7: RANGE versus ROWS  *(guide section 7)*

Orders between `2024-11-22` and `2024-11-26` (inclusive), one row per order, with two running totals of
revenue: one by `ROWS`, one by `RANGE`. Non-cancelled orders only.

Columns `order_id`, `order_date`, `revenue` (rounded to 2), `running_rows`, `running_range` (both rounded to
2). Order by `order_date`, then `order_id`.

Two orders share `2024-11-26` — that is the tie this exercise is testing.


In [53]:
sql = """
with cte as (select o.order_id as order_id , o.order_date as order_date , sum(oi.quantity*oi.unit_price*(1-oi.discount)) as revenue
from orders o join order_items oi 
on o.order_id = oi.order_id 
where o.order_date between '2024-11-22' and '2024-11-26' AND o.status != 'cancelled'
group by o.order_id , o.order_date)
select order_id , order_date , round(revenue , 2) as revenue ,
sum(revenue) over (order by order_date , order_id rows between unbounded preceding and current row) as running_rows,
sum(revenue) over (order by order_date  range between unbounded preceding and current row) as running_range
from cte 


"""

out = q(sql)
print(out)

assert list(out.columns) == ["order_id", "order_date", "revenue", "running_rows", "running_range"]
assert len(out) == 4
assert out.values.tolist() == [
    [267, "2024-11-22", 102600.0, 102600.0, 102600.0],
    [260, "2024-11-23", 105800.0, 208400.0, 208400.0],
    [269, "2024-11-26", 32300.0, 240700.0, 265400.0],
    [271, "2024-11-26", 24700.0, 265400.0, 265400.0],
]
print("OK")


   order_id  order_date   revenue  running_rows  running_range
0       267  2024-11-22  102600.0      102600.0       102600.0
1       260  2024-11-23  105800.0      208400.0       208400.0
2       269  2024-11-26   32300.0      240700.0       265400.0
3       271  2024-11-26   24700.0      265400.0       265400.0
OK


## Exercise 8: PERCENT_RANK and CUME_DIST  *(guide section 8)*

Every product in the **Storage** category (`category_id = 6`), with its price percentile two ways.

Columns `name`, `price`, `pct_rank` (`PERCENT_RANK`, rounded to 4), `cume_dist` (`CUME_DIST`, rounded to 4).
Order by `price`.


In [55]:
sql = """
select name , price , percent_rank() over(order by price) as pct_rank , cume_dist() over(order by price) as cume_dist
from products 
where category_id=6
"""

out = q(sql)
print(out)

assert list(out.columns) == ["name", "price", "pct_rank", "cume_dist"]
assert len(out) == 5
assert out.values.tolist() == [
    ["Carry 128GB Flash Drive", 1150.0, 0.0, 0.2],
    ["Carry 256GB Flash Drive", 1950.0, 0.25, 0.4],
    ["Vault 1TB SSD", 6400.0, 0.5, 0.6],
    ["Vault 4TB HDD", 7300.0, 0.75, 0.8],
    ["Vault 2TB SSD", 11200.0, 1.0, 1.0],
]
print("OK")


                      name    price  pct_rank  cume_dist
0  Carry 128GB Flash Drive   1150.0      0.00        0.2
1  Carry 256GB Flash Drive   1950.0      0.25        0.4
2            Vault 1TB SSD   6400.0      0.50        0.6
3            Vault 4TB HDD   7300.0      0.75        0.8
4            Vault 2TB SSD  11200.0      1.00        1.0
OK


## Exercise 9: Same Question, Three Tools  *(guide section 9)*

The guide found each customer's **most recent** order date three ways. Find each customer's **first** order
date instead, using the window-function approach (`ROW_NUMBER`, not `MAX` or a correlated subquery), then keep
the **10 customers whose first order came earliest**.

Columns `customer_id`, `name`, `first_order_date`. Order by `first_order_date`, then `customer_id`. Limit 10.

Only customers who have actually ordered can appear — there is no "first order date" for the nine who never
ordered, and an `INNER JOIN` (not `LEFT`) from `customers` to your ranked orders is what removes them.


In [65]:
sql = """
with cte as (select c.name as name , o.customer_id as customer_id , o.order_date  as first_order_date, row_number() over (partition by o.customer_id order by o.order_date) as rn
from orders o left join customers c on o.customer_id = c.customer_id)
select customer_id , name ,  first_order_date
from cte 
where rn=1
order by first_order_date , customer_id
limit 10
"""

out = q(sql)
print(out)

assert list(out.columns) == ["customer_id", "name", "first_order_date"]
assert len(out) == 10
assert out.values.tolist() == [
    [34, "Zara Mehta", "2023-01-10"],
    [54, "Mahesh Iyer", "2023-01-10"],
    [6, "Hema Khan", "2023-01-26"],
    [25, "Farah Chopra", "2023-01-30"],
    [14, "Bhavya Pillai", "2023-02-01"],
    [13, "Yamini Reddy", "2023-02-04"],
    [22, "Ojas Nair", "2023-02-10"],
    [38, "Parvati Chopra", "2023-02-12"],
    [9, "Bhavya Menon", "2023-02-14"],
    [18, "Yash Bose", "2023-02-14"],
]
print("OK")


   customer_id            name first_order_date
0           34      Zara Mehta       2023-01-10
1           54     Mahesh Iyer       2023-01-10
2            6       Hema Khan       2023-01-26
3           25    Farah Chopra       2023-01-30
4           14   Bhavya Pillai       2023-02-01
5           13    Yamini Reddy       2023-02-04
6           22       Ojas Nair       2023-02-10
7           38  Parvati Chopra       2023-02-12
8            9    Bhavya Menon       2023-02-14
9           18       Yash Bose       2023-02-14
OK


## Exercise 10: A Date Spine Without Recursion  *(guide section 10)*

Every day from `2023-02-01` to `2023-02-14` inclusive (14 days), with a count of orders placed that day —
**zero**, not a missing row, for the days with none.

Columns `order_date`, `orders`. Order by `order_date`.

Write the 14-day spine as fourteen `UNION ALL SELECT date(...)` lines, exactly as the guide did, then
`LEFT JOIN` it to `orders`.


In [68]:
sql = """
WITH days AS (
    SELECT date('2023-02-01', '+0 days')  AS d UNION ALL SELECT date('2023-02-01', '+1 days')
    UNION ALL SELECT date('2023-02-01', '+2 days') UNION ALL SELECT date('2023-02-01', '+3 days')
    UNION ALL SELECT date('2023-02-01', '+4 days') UNION ALL SELECT date('2023-02-01', '+5 days')
    UNION ALL SELECT date('2023-02-01', '+6 days') UNION ALL SELECT date('2023-02-01', '+7 days')
    UNION ALL SELECT date('2023-02-01', '+8 days') UNION ALL SELECT date('2023-02-01', '+9 days')
    UNION ALL SELECT date('2023-02-01', '+10 days') UNION ALL SELECT date('2023-02-01', '+11 days')
    UNION ALL SELECT date('2023-02-01', '+12 days') UNION ALL SELECT date('2023-02-01', '+13 days')
)
select d.d as order_date , count(o.order_date) as orders
from days d left join orders o 
on d.d = o.order_date
group by d.d
order by d.d
"""

out = q(sql)
print(out)

assert list(out.columns) == ["order_date", "orders"]
assert len(out) == 14
assert out.values.tolist() == [
    ["2023-02-01", 1], ["2023-02-02", 0], ["2023-02-03", 0], ["2023-02-04", 1],
    ["2023-02-05", 0], ["2023-02-06", 0], ["2023-02-07", 0], ["2023-02-08", 0],
    ["2023-02-09", 0], ["2023-02-10", 1], ["2023-02-11", 0], ["2023-02-12", 1],
    ["2023-02-13", 0], ["2023-02-14", 2],
]
assert out["orders"].sum() == 6
print("OK")


    order_date  orders
0   2023-02-01       1
1   2023-02-02       0
2   2023-02-03       0
3   2023-02-04       1
4   2023-02-05       0
5   2023-02-06       0
6   2023-02-07       0
7   2023-02-08       0
8   2023-02-09       0
9   2023-02-10       1
10  2023-02-11       0
11  2023-02-12       1
12  2023-02-13       0
13  2023-02-14       2
OK


## Exercise 11: Refactoring a Subquery Monster  *(guide section 11)*

Below is a working, deeply nested query: employees whose **handled revenue** (from the `store`/`phone` orders
assigned to them) beats the **average handled revenue across all employees who handled at least one order**.
Do not change it — it is given so you can check your answer against it.

Rewrite it as a two-CTE chain — `employee_revenue` (one row per employee, their handled revenue) and
`avg_revenue` (the single average) — that returns the identical rows in the identical order, and prove it with
`.equals()`.


In [ ]:
nested_sql = """
SELECT e.name, e.role,
       ROUND((SELECT SUM(i.quantity * i.unit_price * (1 - i.discount))
              FROM orders o JOIN order_items i ON i.order_id = o.order_id
              WHERE o.employee_id = e.employee_id AND o.status != 'cancelled'), 2) AS handled_revenue
FROM employees e
WHERE (SELECT SUM(i.quantity * i.unit_price * (1 - i.discount))
       FROM orders o JOIN order_items i ON i.order_id = o.order_id
       WHERE o.employee_id = e.employee_id AND o.status != 'cancelled') >
      (SELECT AVG(emp_rev) FROM (
          SELECT SUM(i2.quantity * i2.unit_price * (1 - i2.discount)) AS emp_rev
          FROM orders o2 JOIN order_items i2 ON i2.order_id = o2.order_id
          WHERE o2.employee_id IS NOT NULL AND o2.status != 'cancelled'
          GROUP BY o2.employee_id
      ))
ORDER BY handled_revenue DESC
"""
nested = q(nested_sql)

chained_sql = """
-- Your CTE-chain SQL here
"""
chained = q(chained_sql)

assert list(chained.columns) == ["name", "role", "handled_revenue"]
assert len(chained) == 3
assert chained.values.tolist() == [
    ["Devendra Joshi", "Sales Rep", 2077812.5],
    ["Nikhil Verma", "Sales Rep", 1260915.0],
    ["Arjun Pillai", "Sales Rep", 1001712.5],
]
assert nested.equals(chained), "the refactor must return the identical rows in the identical order"
print("OK")


## Exercise 12: Three Classic Interview Patterns  *(guide section 12)*

Three independent one-off queries.

1. `third_highest` — a single row, single column `price`: the **third**-highest distinct product price,
   without `LIMIT`/`OFFSET`. Nest the "highest that is less than the highest" trick one level deeper.
2. `gaps` — the **5 customers** with the longest gap, in whole days, between two of their own consecutive
   non-cancelled orders. Columns `customer_id`, `longest_gap_days`. Order by `longest_gap_days` descending,
   then `customer_id`.
3. `yoy` — one row comparing the **whole year's** revenue: columns `revenue_2024`, `revenue_2023`
   (both rounded to 2), `yoy_change` (rounded to 2), `yoy_pct` (rounded to 1).


In [89]:
third_sql = """select max(price) as price from products where price <
(select max(price) from products where price < (select max(price) from products))
"""
third_highest = q(third_sql)

gaps_sql = """
with cte as (select customer_id , order_date , lag(order_date) over (partition by customer_id order by order_date) as prev_date
from orders
where status <> 'cancelled')
select customer_id , cast(julianday(order_date)-julianday(prev_date) as integer) as longest_gap_days
from cte where prev_date is not null 
order by longest_gap_days desc , customer_id
limit 5
"""
gaps = q(gaps_sql)

yoy_sql = """
with cte as(select strftime('%Y',o.order_date) as yr , sum(oi.quantity*oi.unit_price*(1-oi.discount)) as revenue
from orders o join order_items oi
on o.order_id = oi.order_id
where o.status <> 'cancelled'
group by yr),
mid as (select sum(case when yr='2023' then revenue end) as revenue_2023,
sum(case when yr='2024' then revenue end ) as revenue_2024
from cte)
select revenue_2024 , revenue_2023 , revenue_2024-revenue_2023 as yoy_change , 
round((revenue_2024-revenue_2023)*100.0/revenue_2023,2) as yoy_pct
from mid

"""
yoy = q(yoy_sql)
print(yoy)

assert list(third_highest.columns) == ["price"]
assert third_highest["price"][0] == 94000.0

assert list(gaps.columns) == ["customer_id", "longest_gap_days"]
assert len(gaps) == 5
assert gaps.values.tolist() == [
    [54, 691], [30, 526], [45, 455], [37, 386], [58, 323],
]

assert list(yoy.columns) == ["revenue_2024", "revenue_2023", "yoy_change", "yoy_pct"]
assert yoy.values.tolist() == [[14378922.5, 7718010.0, 6660912.5, 86.3]]
print("OK")


   revenue_2024  revenue_2023  yoy_change  yoy_pct
0    14378922.5     7718010.0   6660912.5     86.3
OK


## Exercise 13: The LAST_VALUE Trap  *(guide section 14)*

For **customer 3 only**, compute their last order's channel two ways, on one row:

- `wrong_last` — `LAST_VALUE(channel) OVER (PARTITION BY customer_id ORDER BY order_date, order_id)`, no
  explicit frame.
- `right_last` — the same, with `RANGE BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` added.

Columns `customer_id`, `wrong_last`, `right_last`. One row, for `customer_id = 3` only. Non-cancelled orders.

The two columns are expected to **disagree** — that disagreement is what you are checking for.


In [96]:
sql = """
select customer_id , last_value (channel) over(partition by customer_id order by order_date , order_id) as wrong_last , 
last_value (channel) over(partition by customer_id order by order_date , order_id  range between unbounded preceding and unbounded following) as right_last
from orders
where customer_id=3 and status <> 'cancelled'
limit 1
"""

out = q(sql)
print(out)

assert list(out.columns) == ["customer_id", "wrong_last", "right_last"]
assert len(out) == 1
assert out.values.tolist() == [[3, "app", "web"]]
assert out["wrong_last"][0] != out["right_last"][0], "the whole point is that these disagree"
print("OK")


   customer_id wrong_last right_last
0            3        app        web
OK


## Exercise 14: Mini Project — A Recommendation Feed  *(mini project)*

For every product that has ever shared an order with another product, find its single best-selling **partner**
and what share of its total pairings that partner accounts for.

Columns:

- `product`
- `best_partner` — the product most often bought alongside it
- `times_together`
- `partner_share_pct` — `times_together` as a percentage of this product's pairings **with every partner it
  has ever had**, rounded to 1

Order by `times_together` descending, then `product`. Limit 15.

Three CTEs: unordered `counted` pairs (both directions, `<>`), a `totals` CTE summing every partner's count
per product, and a `ranked` CTE picking the top partner per product with `ROW_NUMBER`.


In [115]:
sql = """
WITH cte AS (
    SELECT
        oi.order_id,
        oi.product_id,
        p.name
    FROM order_items oi
    JOIN products p
        ON oi.product_id = p.product_id
),

counted AS (
    SELECT
        c.product_id,
        p.product_id AS partner_id,
        COUNT(*) AS times_together
    FROM cte c
    JOIN cte p
        ON c.order_id = p.order_id
       AND c.product_id <> p.product_id
    GROUP BY c.product_id, p.product_id
),

totals AS (
    SELECT
        product_id,
        SUM(times_together) AS total_pairings
    FROM counted
    GROUP BY product_id
),

ranked AS (
    SELECT
        c.product_id,
        c.partner_id,
        c.times_together,
        ROUND(
            c.times_together * 100.0 / t.total_pairings,
            1
        ) AS partner_share_pct,
        ROW_NUMBER() OVER (
            PARTITION BY c.product_id
            ORDER BY c.times_together DESC, c.partner_id
        ) AS rn
    FROM counted c
    JOIN totals t
        ON c.product_id = t.product_id
)

SELECT
    p.name AS product,
    pp.name AS best_partner,
    r.times_together,
    r.partner_share_pct
FROM ranked r
JOIN products p
    ON r.product_id = p.product_id
JOIN products pp
    ON r.partner_id = pp.product_id
WHERE r.rn = 1
ORDER BY r.times_together DESC, product
LIMIT 15;
"""

out = q(sql)
print(out)
assert list(out.columns) == ["product", "best_partner", "times_together", "partner_share_pct"]
assert len(out) == 15
assert out.values.tolist() == [
    ["Anchor 100W Charger", "Braid USB-C Cable 1m", 4, 8.7],
    ["Braid USB-C Cable 1m", "Anchor 100W Charger", 4, 10.0],
    ["Clarity 24 Monitor", "Braid USB-C Cable 1m", 4, 8.7],
    ["Clarity 32 4K Monitor", "Vault 2TB SSD", 4, 10.8],
    ["Echo Buds Pro", "Rumble Bluetooth Speaker", 4, 11.4],
    ["Frame 50mm Lens", "Frame Compact Camera", 4, 10.8],
    ["Frame Compact Camera", "Frame 50mm Lens", 4, 11.4],
    ["Quiet Desk Mic", "Vault 4TB HDD", 4, 13.8],
    ["Rumble Bluetooth Speaker", "Echo Buds Pro", 4, 15.4],
    ["Vault 1TB SSD", "Vault 4TB HDD", 4, 8.9],
    ["Vault 2TB SSD", "Clarity 32 4K Monitor", 4, 8.7],
    ["Vault 4TB HDD", "Quiet Desk Mic", 4, 9.1],
    ["Aster 14 Laptop", "Orbit Pro Phone", 3, 15.8],
    ["Aster 15 Pro Laptop", "Echo Buds Pro", 3, 12.5],
    ["Carry 256GB Flash Drive", "Tick Smartwatch", 3, 10.7],
]
print("OK")


                     product              best_partner  times_together  partner_share_pct
0        Anchor 100W Charger      Braid USB-C Cable 1m               4                8.7
1       Braid USB-C Cable 1m       Anchor 100W Charger               4               10.0
2         Clarity 24 Monitor      Braid USB-C Cable 1m               4                8.7
3      Clarity 32 4K Monitor             Vault 2TB SSD               4               10.8
4              Echo Buds Pro  Rumble Bluetooth Speaker               4               11.4
5            Frame 50mm Lens      Frame Compact Camera               4               10.8
6       Frame Compact Camera           Frame 50mm Lens               4               11.4
7             Quiet Desk Mic             Vault 4TB HDD               4               13.8
8   Rumble Bluetooth Speaker             Echo Buds Pro               4               15.4
9              Vault 1TB SSD             Vault 4TB HDD               4                8.9
10        

## Exercise 15: Mini Project — A Customer Health Scorecard  *(mini project)*

One row per customer, for the **top 15 by lifetime value**, combining several of this level's techniques.

Columns:

- `name`
- `ltv` — lifetime value, rounded to 2
- `ltv_percentile` — `PERCENT_RANK()` over lifetime value across **every** customer who has ordered, as a
  percentage rounded to 1
- `first_channel`, `last_channel` — using `FIRST_VALUE`/`LAST_VALUE` with the correct frame
- `channel_switched` — `'yes'` or `'no'`
- `longest_gap_days` — the longest gap between two consecutive orders, `0` for a customer with only one order

Order by `ltv` descending. Limit 15.

Build it as a chain of CTEs: lifetime value per customer, the percentile over that, first/last channel per
customer (with the frame from section 6), and the longest gap (from section 12) — then join all four together.
A customer with exactly one order has no gap to compute, which is why `longest_gap_days` needs a `LEFT JOIN`
and a `COALESCE` to `0` rather than an inner join.


In [144]:
sql = """
WITH ltvtable AS (
    SELECT
        c.name AS name,
        c.customer_id AS cust_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) AS revenue
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    LEFT JOIN customers c
        ON o.customer_id = c.customer_id
    WHERE o.status <> 'cancelled'
    GROUP BY o.customer_id
),

rnked AS (
    SELECT
        c.customer_id,
        FIRST_VALUE(o.channel) OVER (
            PARTITION BY c.customer_id
            ORDER BY o.order_date, o.order_id
        ) AS first_value,

        LAST_VALUE(o.channel) OVER (
            PARTITION BY c.customer_id
            ORDER BY o.order_date, o.order_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_value,

        ROW_NUMBER() OVER (
            PARTITION BY c.customer_id
            ORDER BY o.order_date, o.order_id
        ) AS rn
    FROM customers c
    LEFT JOIN orders o
        ON o.customer_id = c.customer_id
    WHERE o.status <> 'cancelled'
),

gaps AS (
    SELECT
        customer_id,
        order_date,
        LAG(order_date) OVER (
            PARTITION BY customer_id
            ORDER BY order_date, order_id
        ) AS previous_order_date
    FROM orders
    WHERE status <> 'cancelled'
),

longest_gaps AS (
    SELECT
        customer_id,
        COALESCE(
            MAX(
                julianday(order_date) -
                julianday(previous_order_date)
            ),
            0
        ) AS longest_gap_days
    FROM gaps
    GROUP BY customer_id
),

mid AS (
    SELECT
        l.cust_id AS customer_id,
        l.name AS name,
        l.revenue AS ltv,
        r.first_value,
        r.last_value,
        PERCENT_RANK() OVER (
    ORDER BY l.revenue
) * 100 as ltv_percent,
        CASE
            WHEN r.first_value = r.last_value THEN 'no'
            ELSE 'yes'
        END AS channel_switched,
        COALESCE(g.longest_gap_days, 0) AS longest_gap_days
    FROM ltvtable l
    LEFT JOIN rnked r
        ON l.cust_id = r.customer_id
       AND r.rn = 1
    LEFT JOIN longest_gaps g
        ON l.cust_id = g.customer_id
)

SELECT
    name,
    ROUND(ltv, 2) AS ltv,
    ltv_percent as ltv_percentile,
    first_value as first_channel,
    last_value as last_channel,
    channel_switched,
    cast(longest_gap_days as integer) as longest_gap_days
FROM mid
ORDER BY ltv DESC
LIMIT 15;

"""

out = q(sql)
print(out)
assert list(out.columns) == ["name", "ltv", "ltv_percentile", "first_channel", "last_channel", "channel_switched", "longest_gap_days"]
assert len(out) == 15
assert out.values.tolist() == [
    ["Hema Khan", 2835360.0, 100.0, "web", "web", "no", 305],
    ["Neha Reddy", 1539182.5, 98.0, "store", "app", "yes", 124],
    ["Zara Mehta", 1257470.0, 96.0, "app", "app", "no", 185],
    ["Varun Pillai", 1002250.0, 94.0, "app", "store", "yes", 187],
    ["Parvati Chopra", 826850.0, 92.0, "web", "phone", "yes", 166],
    ["Nisha Nair", 805260.0, 90.0, "store", "store", "no", 122],
    ["Yash Bose", 791490.0, 88.0, "web", "store", "yes", 247],
    ["Janaki Patel", 783195.0, 86.0, "app", "app", "no", 137],
    ["Manoj Menon", 781007.5, 84.0, "web", "web", "no", 144],
    ["Harish Reddy", 759455.0, 82.0, "store", "app", "yes", 176],
    ["Bhavya Menon", 738960.0, 80.0, "phone", "app", "yes", 165],
    ["Bhavya Das", 648580.0, 78.0, "phone", "web", "yes", 269],
    ["Lalita Mehta", 551370.0, 76.0, "app", "web", "yes", 118],
    ["Farah Chopra", 540040.0, 74.0, "web", "phone", "yes", 247],
    ["Eshan Gupta", 527835.0, 72.0, "app", "app", "no", 316],
]
print("OK")


              name        ltv  ltv_percentile first_channel last_channel channel_switched  longest_gap_days
0        Hema Khan  2835360.0           100.0           web          web               no               305
1       Neha Reddy  1539182.5            98.0         store          app              yes               124
2       Zara Mehta  1257470.0            96.0           app          app               no               185
3     Varun Pillai  1002250.0            94.0           app        store              yes               187
4   Parvati Chopra   826850.0            92.0           web        phone              yes               166
5       Nisha Nair   805260.0            90.0         store        store               no               122
6        Yash Bose   791490.0            88.0           web        store              yes               247
7     Janaki Patel   783195.0            86.0           app          app               no               137
8      Manoj Menon   781007.

## Self-Review Checklist

Check whether you can do each of these **without looking at the guide**.

- [ ] Deduplicate rows that represent the same entity, and justify the tiebreaker you chose.
- [ ] Write a self-join for association (co-occurrence) and explain how its guard differs from a hierarchy
      self-join's.
- [ ] Flatten a hierarchy of known depth with chained self-joins, and say when that stops working.
- [ ] Unpivot columns to rows with `UNION ALL`, and say why every branch needs the same column shape.
- [ ] Explain why `LAST_VALUE` needs an explicit frame when `FIRST_VALUE` does not.
- [ ] Say what `RANGE` does differently from `ROWS` when the `ORDER BY` has ties.
- [ ] Choose between `PERCENT_RANK`, `CUME_DIST` and `NTILE` for a given percentile question.
- [ ] Answer the same question with a correlated subquery, a join, and a window function, and say which is
      likely fastest and which generalises most easily.
- [ ] Build a date spine by hand for a short, fixed range, and say when it should be a recursive CTE instead.
- [ ] Turn a nested subquery into a named CTE chain without changing the answer, and prove it did not change.
- [ ] Find the Nth-highest value without `LIMIT`/`OFFSET`.
- [ ] Compute the longest gap between two events with `LAG` and date arithmetic.
- [ ] Write a year-over-year comparison as a self-join, and say what an inner join would silently drop.

## What Next

`sql-advanced` starts immediately after this: `EXPLAIN QUERY PLAN`, indexes, the recursive CTE that
generalises sections 4 and 10 of this level, transactions, and schema design. Every technique in this notebook
is something `sql-advanced` assumes you can already do without help.

Before you move on, take the two mini projects and change which slice of customers or products they cover. If
the numbers still make sense, you have understood them rather than copied them.
